In [ ]:
#Install Dependencies and prints real errors if something fails.  

import sys, subprocess

def pip_install(packages, label):
    """Install packages and print real errors on failure."""
    print(f"\n Installing {label}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install"] + packages,
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(" INSTALL FAILED — error output below:")
        print(result.stdout[-3000:] if result.stdout else "")
        print(result.stderr[-3000:] if result.stderr else "")
        raise RuntimeError(f"{label} installation failed")
    print(f" {label} installed")

#  Fix httpx FIRST (must be done before openai is loaded) 
# openai 1.54.x uses httpx internals that were removed in httpx>=0.28
# Pinning to 0.27.2 resolves the 'unexpected keyword argument proxies' error
pip_install(["httpx==0.27.2"], "httpx (compatibility fix)")

pip_install([
    "langchain==0.3.7",
    "langchain-community==0.3.7",
    "langchain-core==0.3.19",
    "langchain-openai==0.2.9",
    "langchain-chroma==0.1.4",
    "openai==1.54.3",
    "tiktoken==0.8.0",
    "python-dotenv==1.0.1",
], "LangChain + OpenAI")

pip_install([
    "chromadb==0.5.18",
    "pypdf==4.3.1",
], "Chroma + PyPDF")


In [ ]:

#conversational, agentic RAG system with retrieval-augmented summarization and keyword search tools.
#Imports
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.tools import Tool
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

import ipywidgets as widgets
from IPython.display import display, HTML

print(" All imports successful!")

In [ ]:
#  Configuration

os.environ["OPENAI_API_KEY"] = ""   #  replace

PDF_PATH = r""  #your actual file path

MODEL_NAME    = "gpt-4o" 
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 200
CHROMA_DIR    = "./chroma_db"


In [ ]:
#PDF Loading and Splitting - ingestion

def load_pdf(path: str):
    loader = PyPDFLoader(path)
    return loader.load()

pages = load_pdf(PDF_PATH)

print(f" Pages  : {len(pages)}")


In [ ]:
#chunking with overlap to preserve context across splits
def chunk_documents(pages, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_documents(pages)

chunks = chunk_documents(pages, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"Chunk size    : {CHUNK_SIZE} chars")
print(f"Chunk overlap : {CHUNK_OVERLAP} chars")
print(f"Total chunks  : {len(chunks)}")
print(f"\n--- Sample chunk (page {chunks[0].metadata.get('page','?')}) ---")


In [ ]:
#Initialize embeddings and vector store using OpenAI's text-embedding-3-small model.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Smoke test
test_vec = embeddings.embed_query("hello world")
print(f" Embeddings ready  |  Vector size: {len(test_vec)}")

In [ ]:
# chroma vector store
def build_vector_store(chunks, embeddings, persist_dir):
    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_dir,
    )

print(" Building Chroma vector store...")
vectorstore = build_vector_store(chunks, embeddings, CHROMA_DIR)

count = vectorstore._collection.count()
print(f"Vector store ready  {count} vectors saved to '{CHROMA_DIR}'")

# Similarity search test
print("\n--- Similarity search test ---")
for i, doc in enumerate(vectorstore.similarity_search("main topic", k=2), 1):
    print(f"Result {i} (page {doc.metadata.get('page','?')}): {doc.page_content[:120]}...")

In [ ]:
# Retrieval QA Chain + Conversational Memory
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

def build_retrieval_chain(vectorstore, llm):
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True,
        output_key="answer",
    )
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5, "fetch_k": 10},
    )
    return ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        return_source_documents=True,
        verbose=False,
    )

retrieval_chain = build_retrieval_chain(vectorstore, llm)

# Quick test
test = retrieval_chain.invoke({"question": "What is this document about?"})
print("Retrieval chain ready\n")
print("Q: What is this document about?")
print("A:", test["answer"])
src_pages = sorted({d.metadata.get('page','?') for d in test.get('source_documents',[])})
print(f" Source pages: {src_pages}")

In [ ]:
# Tool Calling 
def build_agent(retrieval_chain, vectorstore, llm):

    #  Tool 1  PDF Q&A 
    def pdf_qa_tool(query: str) -> str:
        result = retrieval_chain.invoke({"question": query})
        answer = result["answer"]
        sources = result.get("source_documents", [])
        if sources:
            pages = sorted({d.metadata.get("page", "?") for d in sources})
            answer += f"\n\n Sources: pages {pages}"
        return answer

    #  Tool 2  Summarize 
    def summarize_document(query: str = "") -> str:
        docs = vectorstore.similarity_search("main topic overview introduction", k=6)
        text = "\n\n".join(d.page_content for d in docs)
        return llm.invoke(f"Summarize in 10 sentences:\n\n{text}").content

    # Tool 3  Keyword Search 
    def keyword_search(keyword: str) -> str:
        docs = vectorstore.similarity_search(keyword, k=4)
        if not docs:
            return f"Nothing found for '{keyword}'."
        return "\n\n".join(
            f"Result {i} (Page {d.metadata.get('page','?')}):\n{d.page_content[:300]}..."
            for i, d in enumerate(docs, 1)
        )

    # Tool 4  List Topics
    def list_topics(query: str = "") -> str:
        docs = vectorstore.similarity_search("topics sections chapters headings", k=8)
        text = "\n".join(d.page_content[:200] for d in docs)
        return llm.invoke(f"List the main topics as bullet points:\n\n{text}").content

    tools = [
        Tool(name="pdf_qa",
             func=pdf_qa_tool,
             description="Answer any specific question about the PDF. Input: a question."),
        Tool(name="summarize_document",
             func=summarize_document,
             description="Get a high-level summary of the document."),
        Tool(name="keyword_search",
             func=keyword_search,
             description="Find a keyword or concept in the document. Input: the keyword."),
        Tool(name="list_topics",
             func=list_topics,
             description="List the main topics covered in the document."),
    ]

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         """You are an intelligent PDF assistant. Use tools to answer questions.
            Always cite page numbers when available. Be concise and accurate.
            Tools: pdf_qa | summarize_document | keyword_search | list_topics"""),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])

    agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=True,
                         max_iterations=5, handle_parsing_errors=True)

agent_executor = build_agent(retrieval_chain, vectorstore, llm)
print(" Agent with 4 tools is ready!")

In [ ]:
chat_history = []

#Widgets Interactive Chat UI
out = widgets.Output(
    layout=widgets.Layout(
        height="420px", overflow_y="auto",
        border="1px solid #ddd", padding="12px", border_radius="10px"
    )
)
txt = widgets.Text(
    placeholder="Ask anything about your PDF… (Enter to send)",
    layout=widgets.Layout(width="76%", height="36px")
)
btn_send  = widgets.Button(description="Send ➤",  button_style="primary",
                            layout=widgets.Layout(width="11%", height="36px"))
btn_clear = widgets.Button(description="🧹 Clear", button_style="warning",
                            layout=widgets.Layout(width="11%", height="36px"))

#  Chat bubble renderer 
def bubble(role, text, bg, align):
    safe = text.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;").replace("\n","<br>")
    with out:
        display(HTML(f"""
        <div style="text-align:{align};margin:6px 0">
          <span style="display:inline-block;background:{bg};border-radius:14px;
                       padding:9px 14px;max-width:85%;font-size:14px;
                       box-shadow:0 1px 3px rgba(0,0,0,.12);">
            <b>{role}</b><br>{safe}
          </span>
        </div>"""))

#  Send handler 
def on_send(_):
    global chat_history
    query = txt.value.strip()
    if not query:
        return
    txt.value = ""
    bubble("You", query, "#DCF8C6", "right")
    with out:
        display(HTML("<i style='color:#888;font-size:12px;margin-left:8px'>🤖 Thinking…</i>"))
    try:
        result = agent_executor.invoke({"input": query, "chat_history": chat_history})
        answer = result["output"]
        chat_history += [HumanMessage(content=query), AIMessage(content=answer)]
        if len(chat_history) > 20:
            chat_history = chat_history[-20:]
        bubble("Assistant", answer, "#F1F0F0", "left")
    except Exception as e:
        bubble("Error", str(e), "#FFE0E0", "left")

#  Clear handler 
def on_clear(_):
    global chat_history
    chat_history = []
    out.clear_output()
    with out:
        display(HTML("<i style='color:#888;padding:8px'>🧹 History cleared — start fresh!</i>"))

btn_send.on_click(on_send)
btn_clear.on_click(on_clear)
txt.on_submit(on_send)

#  Welcome message 
with out:
    display(HTML("""
    <div style="text-align:center;color:#555;padding:24px">
      <h3 style="margin-bottom:8px"> PDF QA Chat </h3>
      <p>Your PDF is loaded. Ask me anything!</p>
      <p style="color:#888;font-size:13px">
        💡 Try:
        <i>"What topics does it cover?"</i> ·
        <i>"Find anything about [keyword]"</i>
      </p>
    </div>"""))

display(out, widgets.HBox([txt, btn_send, btn_clear]))